In [ ]:
import base64
import io
import json
import uuid

import networkx as nx
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import widgets, Tab
from IPython.display import display, clear_output, Javascript

In [ ]:
EXIT_FILE = "../data/test_exits.json"
ROOM_FILE = "../data/test_rooms.json"


class RoomEditorError(Exception):
    """Custom exception for Room Editor operations."""

    pass


class DataManager:
    """Manages room and exit data with proper validation and error handling."""

    def __init__(self, room_file: str, exit_file: str):
        """Initialize the data manager with file paths.

        Args:
            room_file: Path to the rooms JSON file
            exit_file: Path to the exits JSON file
        """
        self.room_file = room_file
        self.exit_file = exit_file
        self.rooms_data = self._load_json_data(room_file, "rooms")
        self.exits_data = self._load_json_data(exit_file, "exits")

    def _load_json_data(self, filename: str, data_type: str) -> dict:
        """Load and validate JSON data from file.

        Args:
            filename: Path to the JSON file
            data_type: Type of data being loaded ('rooms' or 'exits')

        Returns:
            Dictionary containing the loaded data

        Raises:
            RoomEditorError: If file cannot be loaded or parsed
        """
        try:
            with open(filename, "r", encoding="utf-8") as file:
                data = json.load(file)

            if not isinstance(data, dict):
                raise RoomEditorError(f"Invalid JSON structure in {filename}: expected object")

            if data_type not in data:
                raise RoomEditorError(f"Missing '{data_type}' key in {filename}")

            if not isinstance(data[data_type], list):
                raise RoomEditorError(f"Invalid data structure: '{data_type}' must be a list")

            return data

        except FileNotFoundError:
            print(f"Warning: File {filename} not found, creating empty data structure")
            return {data_type: []}
        except json.JSONDecodeError:
            raise RoomEditorError(f"Invalid JSON in {filename}")
        except Exception as err:
            raise RoomEditorError(f"Unexpected error loading {filename}:{err}")

    def validate_room_data(self, room: dict) -> bool:
        """Validate room data structure.

        Args:
            room: Room dictionary to validate

        Returns:
            True if valid, False otherwise
        """
        required_fields = {"RoomID", "Title"}
        if not isinstance(room, dict):
            return False
        if not all(field in room for field in required_fields):
            return False
        if not isinstance(room["RoomID"], int) or room["RoomID"] <= 0:
            return False
        if not isinstance(room["Title"], str) or not room["Title"].strip():
            return False
        return True

    def validate_exit_data(self, exit: dict) -> bool:
        """Validate exit data structure.

        Args:
            exit: Exit dictionary to validate

        Returns:
            True if valid, False otherwise
        """
        required_fields = {"ExitID", "Direction", "TargetRoom", "Visible"}
        if not isinstance(exit, dict):
            return False
        if not all(field in exit for field in required_fields):
            return False
        if not isinstance(exit["ExitID"], str) or not exit["ExitID"].strip():
            return False
        if not isinstance(exit["Direction"], str) or not exit["Direction"].strip():
            return False
        if not isinstance(exit["TargetRoom"], int) or exit["TargetRoom"] <= 0:
            return False
        if not isinstance(exit["Visible"], bool):
            return False
        return True

    def save_data(self) -> None:
        """Save both rooms and exits data to files.

        Raises:
            RoomEditorError: If files cannot be saved
        """
        try:
            with open(self.exit_file, "w", encoding="utf-8") as f:
                json.dump(self.exits_data, f, indent=2)
            with open(self.room_file, "w", encoding="utf-8") as f:
                json.dump(self.rooms_data, f, indent=2)
        except Exception as err:
            raise RoomEditorError(f"Error saving files:{err}")

    def find_room_by_id(self, room_id: int):
        """Find a room by its ID.

        Args:
            room_id: ID of the room to find

        Returns:
            Room dictionary if found, None otherwise
        """
        return next((room for room in self.rooms_data["rooms"] if room["RoomID"] == room_id), None)

    def find_exit_by_id(self, exit_id: str):
        """Find an exit by its ID.

        Args:
            exit_id: ID of the exit to find

        Returns:
            Exit dictionary if found, None otherwise
        """
        return next((exit for exit in self.exits_data["exits"] if exit["ExitID"] == exit_id), None)

    def validate_target_room(self, target_room_id: int) -> bool:
        """Validate that a target room exists.

        Args:
            target_room_id: ID of the target room

        Returns:
            True if room exists, False otherwise
        """
        return any(room["RoomID"] == target_room_id for room in self.rooms_data["rooms"])

    def get_next_room_id(self) -> int:
        """Get the next available room ID.

        Returns:
            Next available room ID
        """
        existing_ids = {room["RoomID"] for room in self.rooms_data["rooms"]}
        new_id = 1
        while new_id in existing_ids:
            new_id += 1
        return new_id

    def remove_exit_from_rooms(self, exit_id: str) -> None:
        """Remove an exit ID from all rooms that reference it.

        Args:
            exit_id: ID of the exit to remove
        """
        for room in self.rooms_data["rooms"]:
            if "ExitID" in room and exit_id in room["ExitID"]:
                room["ExitID"].remove(exit_id)

    def get_orphaned_exits(self) -> list:
        """Find exits that are not referenced by any room.

        Returns:
            List of orphaned exit dictionaries
        """
        referenced_exit_ids = set()
        for room in self.rooms_data["rooms"]:
            referenced_exit_ids.update(room.get("ExitID", []))

        orphaned_exits = []
        for exit in self.exits_data["exits"]:
            if exit["ExitID"] not in referenced_exit_ids:
                orphaned_exits.append(exit)

        return orphaned_exits

    def cleanup_orphaned_exits(self) -> int:
        """Remove exits that are not referenced by any room.

        Returns:
            Number of exits removed
        """
        orphaned_exits = self.get_orphaned_exits()
        orphaned_count = len(orphaned_exits)

        if orphaned_count > 0:
            orphaned_ids = {exit["ExitID"] for exit in orphaned_exits}
            self.exits_data["exits"] = [exit for exit in self.exits_data["exits"] if exit["ExitID"] not in orphaned_ids]

        return orphaned_count

    def validate_data_consistency(self) -> dict:
        """Validate data consistency and return issues found.

        Returns:
            Dictionary containing lists of consistency issues
        """
        issues = {
            "orphaned_exits": [],
            "invalid_target_rooms": [],
            "duplicate_room_ids": [],
            "duplicate_exit_ids": [],
            "invalid_room_data": [],
            "invalid_exit_data": [],
        }

        # Check for orphaned exits
        issues["orphaned_exits"] = self.get_orphaned_exits()

        # Check for exits with invalid target rooms
        for exit in self.exits_data["exits"]:
            if not self.validate_exit_data(exit):
                issues["invalid_exit_data"].append(exit)
            elif not self.validate_target_room(exit["TargetRoom"]):
                issues["invalid_target_rooms"].append(exit)

        # Check for duplicate room IDs
        room_ids = [room["RoomID"] for room in self.rooms_data["rooms"]]
        seen_room_ids = set()
        for room_id in room_ids:
            if room_id in seen_room_ids:
                issues["duplicate_room_ids"].append(room_id)
            else:
                seen_room_ids.add(room_id)

        # Check for duplicate exit IDs
        exit_ids = [exit["ExitID"] for exit in self.exits_data["exits"]]
        seen_exit_ids = set()
        for exit_id in exit_ids:
            if exit_id in seen_exit_ids:
                issues["duplicate_exit_ids"].append(exit_id)
            else:
                seen_exit_ids.add(exit_id)

        # Check for invalid room data
        for room in self.rooms_data["rooms"]:
            if not self.validate_room_data(room):
                issues["invalid_room_data"].append(room)

        return issues

    def repair_data_consistency(self) -> dict:
        """Attempt to repair data consistency issues.

        Returns:
            Dictionary containing repair results
        """
        results = {
            "orphaned_exits_removed": 0,
            "invalid_target_rooms_fixed": 0,
            "duplicate_room_ids_fixed": 0,
            "duplicate_exit_ids_fixed": 0,
            "invalid_data_removed": 0,
        }

        # Remove orphaned exits
        results["orphaned_exits_removed"] = self.cleanup_orphaned_exits()

        # Fix exits with invalid target rooms by pointing them to room 1
        valid_rooms = [room["RoomID"] for room in self.rooms_data["rooms"] if self.validate_room_data(room)]
        fallback_room = valid_rooms[0] if valid_rooms else 1

        for exit in self.exits_data["exits"]:
            if self.validate_exit_data(exit) and not self.validate_target_room(exit["TargetRoom"]):
                exit["TargetRoom"] = fallback_room
                results["invalid_target_rooms_fixed"] += 1

        # Remove duplicate room IDs (keep first occurrence)
        seen_room_ids = set()
        filtered_rooms = []
        for room in self.rooms_data["rooms"]:
            if room["RoomID"] not in seen_room_ids:
                seen_room_ids.add(room["RoomID"])
                filtered_rooms.append(room)
            else:
                results["duplicate_room_ids_fixed"] += 1
        self.rooms_data["rooms"] = filtered_rooms

        # Remove duplicate exit IDs (keep first occurrence)
        seen_exit_ids = set()
        filtered_exits = []
        for exit in self.exits_data["exits"]:
            if exit["ExitID"] not in seen_exit_ids:
                seen_exit_ids.add(exit["ExitID"])
                filtered_exits.append(exit)
            else:
                results["duplicate_exit_ids_fixed"] += 1
        self.exits_data["exits"] = filtered_exits

        # Remove invalid data
        valid_rooms = []
        for room in self.rooms_data["rooms"]:
            if self.validate_room_data(room):
                valid_rooms.append(room)
            else:
                results["invalid_data_removed"] += 1
        self.rooms_data["rooms"] = valid_rooms

        valid_exits = []
        for exit in self.exits_data["exits"]:
            if self.validate_exit_data(exit):
                valid_exits.append(exit)
            else:
                results["invalid_data_removed"] += 1
        self.exits_data["exits"] = valid_exits

        return results


# Initialize the data manager
try:
    data_manager = DataManager(ROOM_FILE, EXIT_FILE)
except RoomEditorError:
    print("Critical error initializing data manager")
    raise

In [ ]:
# Legacy support - remove this cell after refactoring is complete
def load_json_data(filename: str) -> dict:
    """Legacy function for backward compatibility.

    Args:
        filename: Path to the JSON file

    Returns:
        Dictionary containing the loaded data
    """
    data_type = "exits" if "exits" in filename else "rooms"
    return data_manager._load_json_data(filename, data_type)


# Provide access to data through the data manager
rooms_data = data_manager.rooms_data
exits_data = data_manager.exits_data

In [ ]:
def create_graph() -> nx.MultiDiGraph:
    """Create a NetworkX graph representation of rooms and exits.

    Returns:
        NetworkX MultiDiGraph with rooms as nodes and exits as edges

    Raises:
        RoomEditorError: If graph creation fails due to data inconsistencies
    """
    try:
        # First, check and repair data consistency
        issues = data_manager.validate_data_consistency()
        if any(issues.values()):
            print("Data consistency issues detected. Running automated repair...")
            repair_results = data_manager.repair_data_consistency()

            if repair_results["orphaned_exits_removed"] > 0:
                print(f"Removed {repair_results['orphaned_exits_removed']} orphaned exits")
            if repair_results["invalid_target_rooms_fixed"] > 0:
                print(f"Fixed {repair_results['invalid_target_rooms_fixed']} exits with invalid target rooms")
            if repair_results["duplicate_room_ids_fixed"] > 0:
                print(f"Fixed {repair_results['duplicate_room_ids_fixed']} duplicate room IDs")
            if repair_results["duplicate_exit_ids_fixed"] > 0:
                print(f"Fixed {repair_results['duplicate_exit_ids_fixed']} duplicate exit IDs")
            if repair_results["invalid_data_removed"] > 0:
                print(f"Removed {repair_results['invalid_data_removed']} invalid data entries")

        G = nx.MultiDiGraph()

        # Create mapping of all valid exit IDs to their data
        exit_map = {exit["ExitID"]: exit for exit in data_manager.exits_data["exits"]}

        # Track room-exit relationships for validation
        room_exit_mapping = {}

        # First pass: Add all rooms and build exit mapping
        for room in data_manager.rooms_data["rooms"]:
            if not data_manager.validate_room_data(room):
                print(f"Warning: Invalid room data for room {room.get('RoomID', 'unknown')}")
                continue

            G.add_node(room["RoomID"], title=room["Title"])
            for exit_id in room.get("ExitID", []):
                if exit_id in exit_map:
                    room_exit_mapping[exit_id] = room["RoomID"]

        # Second pass: Add exits with validation
        for exit_id, exit_data in exit_map.items():
            if not data_manager.validate_exit_data(exit_data):
                print(f"Warning: Invalid exit data for exit {exit_id}")
                continue

            source_room = room_exit_mapping.get(exit_id)
            if source_room is None:
                continue

            # Validate target room exists
            if not data_manager.validate_target_room(exit_data["TargetRoom"]):
                print(f"Warning: Exit {exit_id} targets non-existent room {exit_data['TargetRoom']}")
                continue

            # If exit has explicit RoomID, validate it matches
            if "RoomID" in exit_data and exit_data["RoomID"] != source_room:
                print(f"Warning: Exit {exit_id} has RoomID {exit_data['RoomID']} but is referenced by room {source_room}")

            G.add_edge(source_room, exit_data["TargetRoom"], key=exit_id, direction=exit_data["Direction"])

        return G

    except Exception as err:
        raise RoomEditorError(f"Failed to create graph:{err}")

In [ ]:
def draw_nodes_and_labels(G: nx.MultiDiGraph, pos, ax) -> None:
    """Draw nodes and their labels on the graph.

    Args:
        G: NetworkX graph containing rooms and exits
        pos: Dictionary of node positions
        ax: Matplotlib axis to draw on
    """
    try:
        nx.draw_networkx_nodes(G, pos, node_color="lightblue", node_size=700, ax=ax)
        node_labels = {node: G.nodes[node]["title"] for node in G.nodes()}
        nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=10, ax=ax)
    except Exception as err:
        print(f"Error drawing nodes:{err}")


def calculate_edge_positions(edge_list: list) -> dict:
    """Calculate edge counts for proper positioning.

    Args:
        edge_list: List of edges with their data

    Returns:
        Dictionary mapping edge pairs to their count
    """
    edge_counts = {}
    for u, v, key, data in edge_list:
        edge_counts[(u, v)] = edge_counts.get((u, v), 0) + 1
    return edge_counts


def draw_edges_with_labels(G: nx.MultiDiGraph, pos, edge_list: list, edge_counts: dict, ax) -> None:
    """Draw edges with directional labels.

    Args:
        G: NetworkX graph containing rooms and exits
        pos: Dictionary of node positions
        edge_list: List of edges to draw
        edge_counts: Dictionary of edge counts for positioning
        ax: Matplotlib axis to draw on
    """
    try:
        for (u, v), count in edge_counts.items():
            edges = [(u, v2, k, d) for (u, v2, k, d) in edge_list if v2 == v]
            num_edges = len(edges)
            rad_list = [0.0] if num_edges == 1 else np.linspace(-0.5, 0.5, num_edges)

            for (u, v, key, data), rad in zip(edges, rad_list):
                nx.draw_networkx_edges(
                    G, pos, edgelist=[(u, v, key)], connectionstyle=f"arc3,rad={rad}", arrowsize=15, edge_color="gray", ax=ax
                )

                x1, y1 = pos[u]
                x2, y2 = pos[v]
                xm, ym = (x1 + x2) / 2, (y1 + y2) / 2

                dx, dy = x2 - x1, y2 - y1
                angle = np.arctan2(dy, dx)

                horizontal_scale = 0.05
                vertical_offset = rad * 0.1

                label_x = xm - horizontal_scale * np.sin(angle)
                label_y = ym + vertical_offset * np.cos(angle)

                label = data.get("direction", "No Label")
                ax.text(
                    label_x,
                    label_y,
                    label,
                    fontsize=8,
                    ha="center",
                    va="center",
                    bbox=dict(facecolor="white", edgecolor="none", pad=1),
                )
    except Exception as err:
        print(f"Error drawing edges:{err}")


def visualize_graph(G: nx.MultiDiGraph) -> str:
    """Create a visualization of the room graph.

    Args:
        G: NetworkX graph to visualize

    Returns:
        Base64 encoded string of the graph image

    Raises:
        RoomEditorError: If visualization fails
    """
    try:
        fig, ax = plt.subplots(figsize=(14, 10))
        pos = nx.spring_layout(G, k=1.5, iterations=200)

        draw_nodes_and_labels(G, pos, ax)

        edge_list = list(G.edges(keys=True, data=True))
        edge_counts = calculate_edge_positions(edge_list)
        draw_edges_with_labels(G, pos, edge_list, edge_counts, ax)

        ax.set_title("Room Layout Graph")
        ax.axis("off")

        # Convert plot to image
        buf = io.BytesIO()
        plt.savefig(buf, format="png")
        buf.seek(0)
        img_str = base64.b64encode(buf.read()).decode("utf-8")
        plt.close(fig)

        return img_str

    except Exception as err:
        plt.close("all")
        raise RoomEditorError(f"Failed to create visualization:{err}")

In [ ]:
def create_exit_editor_widgets() -> tuple:
    """Create widgets for exit editing functionality.

    Returns:
        Tuple containing all exit editor widgets
    """
    try:
        exit_list = widgets.Select(
            options=[
                (f"{exit['ExitID']}: {exit['Direction']} to {exit['TargetRoom']}", exit["ExitID"])
                for exit in data_manager.exits_data["exits"]
            ],
            description="Exits:",
            layout=widgets.Layout(width="300px"),
        )

        exit_id = widgets.Text(description="Exit ID:", disabled=True)
        direction = widgets.Text(description="Direction:")
        target_room = widgets.IntText(description="Target Room:")
        visible = widgets.Checkbox(description="Visible", value=True)

        new_button = widgets.Button(description="New Exit")
        save_button = widgets.Button(description="Save Exit")
        delete_button = widgets.Button(description="Delete Exit")
        save_all_button = widgets.Button(description="Save All Exits")

        output = widgets.Output()

        return (
            exit_list,
            exit_id,
            direction,
            target_room,
            visible,
            new_button,
            save_button,
            delete_button,
            save_all_button,
            output,
        )

    except Exception as err:
        print(f"Error creating exit editor widgets:{err}")
        return tuple([widgets.HTML(f"Error:{err}") for _ in range(10)])

In [ ]:
def on_exit_select(
    change: dict, exit_id: widgets.Text, direction: widgets.Text, target_room: widgets.IntText, visible: widgets.Checkbox
) -> None:
    """Handle exit selection from the list.

    Args:
        change: Widget change event data
        exit_id: Exit ID text widget
        direction: Direction text widget
        target_room: Target room integer widget
        visible: Visibility checkbox widget
    """
    try:
        if change["type"] == "change" and change["name"] == "value":
            selected_exit = data_manager.find_exit_by_id(change["new"])
            if selected_exit and data_manager.validate_exit_data(selected_exit):
                exit_id.value = selected_exit["ExitID"]
                direction.value = selected_exit["Direction"]
                target_room.value = selected_exit["TargetRoom"]
                visible.value = selected_exit["Visible"]
            else:
                print(f"Warning: Selected exit {change['new']} not found or invalid")
    except Exception as err:
        print(f"Error selecting exit:{err}")

In [ ]:
def update_exit_list(exit_list: widgets.Select) -> None:
    """Update the exit list widget with current exit data.

    Args:
        exit_list: Exit selection widget to update
    """
    try:
        # Create a set of valid exit IDs from rooms
        valid_exit_ids = set()
        for room in data_manager.rooms_data["rooms"]:
            valid_exit_ids.update(room.get("ExitID", []))

        # Format exit options, only including exits referenced by rooms
        formatted_exits = []
        for exit in data_manager.exits_data["exits"]:
            if not data_manager.validate_exit_data(exit):
                continue

            if exit["ExitID"] in valid_exit_ids:
                # Get source room for context
                source_room = next(
                    (room["RoomID"] for room in data_manager.rooms_data["rooms"] if exit["ExitID"] in room.get("ExitID", [])), None
                )

                # Build display string
                display_text = f"{exit['ExitID']}: "
                display_text += f"From Room {source_room} " if source_room else "Unattached "
                display_text += f"{exit['Direction']} to Room {exit['TargetRoom']}"

                if not exit.get("Visible", True):
                    display_text += " (hidden)"

                formatted_exits.append((display_text, exit["ExitID"]))

        # Sort by source room and direction for easier navigation
        formatted_exits.sort(key=lambda x: x[0])

        # Update the widget options
        exit_list.options = formatted_exits

        # If there are options and no current selection, select the first one
        if formatted_exits and not exit_list.value:
            exit_list.value = formatted_exits[0][1]

    except Exception as err:
        print(f"Error updating exit list:{err}")

In [ ]:
def validate_target_room(target_room_id: int) -> bool:
    """Validate that a target room exists in the rooms data.

    Args:
        target_room_id: ID of the target room to validate

    Returns:
        True if room exists, False otherwise
    """
    return data_manager.validate_target_room(target_room_id)


def find_room_by_id(room_id: int):
    """Find a room by its ID.

    Args:
        room_id: ID of the room to find

    Returns:
        Room dictionary if found, None otherwise
    """
    return data_manager.find_room_by_id(room_id)


def on_new_exit(
    b: widgets.Button,
    exit_list: widgets.Select,
    exit_id: widgets.Text,
    direction: widgets.Text,
    target_room: widgets.IntText,
    visible: widgets.Checkbox,
) -> None:
    """Create a new exit.

    Args:
        b: Button widget that triggered the event
        exit_list: Exit selection widget
        exit_id: Exit ID text widget
        direction: Direction text widget
        target_room: Target room integer widget
        visible: Visibility checkbox widget
    """
    try:
        # Validate target room exists if specified
        if target_room.value != 0 and not data_manager.validate_target_room(target_room.value):
            print(f"Error: Target room {target_room.value} does not exist. Cannot create exit.")
            return

        new_exit = {
            "ExitID": str(uuid.uuid4()),
            "Direction": "",
            "TargetRoom": target_room.value if target_room.value != 0 else 1,
            "Visible": True,
        }

        if not data_manager.validate_exit_data(new_exit):
            # Fix validation issues
            if new_exit["TargetRoom"] <= 0:
                new_exit["TargetRoom"] = 1

        data_manager.exits_data["exits"].append(new_exit)

        # Find the room that this exit belongs to
        if target_room.value != 0:
            target_room_obj = data_manager.find_room_by_id(target_room.value)
            if target_room_obj:
                if "ExitID" not in target_room_obj:
                    target_room_obj["ExitID"] = []
                target_room_obj["ExitID"].append(new_exit["ExitID"])
            else:
                print(f"Warning: Room with ID {target_room.value} not found. Exit not added to any room.")

        update_exit_list(exit_list)
        exit_list.value = new_exit["ExitID"]
        on_exit_select({"type": "change", "name": "value", "new": new_exit["ExitID"]}, exit_id, direction, target_room, visible)

    except Exception as err:
        print(f"Error creating new exit:{err}")

In [ ]:
def on_save_exit(
    b: widgets.Button,
    exit_list: widgets.Select,
    exit_id: widgets.Text,
    direction: widgets.Text,
    target_room: widgets.IntText,
    visible: widgets.Checkbox,
    output: widgets.Output,
) -> None:
    """Save changes to the selected exit.

    Args:
        b: Button widget that triggered the event
        exit_list: Exit selection widget
        exit_id: Exit ID text widget
        direction: Direction text widget
        target_room: Target room integer widget
        visible: Visibility checkbox widget
        output: Output widget for displaying messages
    """
    try:
        selected_exit = data_manager.find_exit_by_id(exit_id.value)
        if not selected_exit:
            with output:
                clear_output()
                print(f"Error: Exit {exit_id.value} not found.")
            return

        # Validate target room exists
        if not data_manager.validate_target_room(target_room.value):
            with output:
                clear_output()
                print(f"Error: Target room {target_room.value} does not exist. Cannot save exit.")
            return

        # Validate direction is not empty
        if not direction.value.strip():
            with output:
                clear_output()
                print("Error: Direction cannot be empty.")
            return

        selected_exit["Direction"] = direction.value.strip()
        selected_exit["TargetRoom"] = target_room.value
        selected_exit["Visible"] = visible.value

        if not data_manager.validate_exit_data(selected_exit):
            with output:
                clear_output()
                print("Error: Exit data validation failed after update.")
            return

        update_exit_list(exit_list)
        with output:
            clear_output()
            print(f"Exit {exit_id.value} updated successfully.")

    except Exception as err:
        with output:
            clear_output()
            print(f"Error saving exit:{err}")

In [ ]:
def remove_exit_from_rooms(exit_id: str) -> None:
    """Remove an exit ID from all rooms that reference it.

    Args:
        exit_id: ID of the exit to remove
    """
    data_manager.remove_exit_from_rooms(exit_id)


def on_save_all(_: widgets.Button, output: widgets.Output) -> None:
    """Save all rooms and exits to their respective files.

    Args:
        _: Button widget that triggered the event (unused)
        output: Output widget for displaying messages
    """
    try:
        data_manager.save_data()
        with output:
            clear_output()
            print("All exits and rooms saved successfully.")
    except RoomEditorError:
        with output:
            clear_output()
            print("Error saving files")
    except Exception as err:
        with output:
            clear_output()
            print(f"Unexpected error saving files:{err}")


def exit_app(_: widgets.Button) -> None:
    """Exit the application safely.

    Args:
        _: Button widget that triggered the event (unused)
    """
    try:
        display(Javascript('window.parent.postMessage({"jupyter_voila_exit": true}, "*")'))
    except Exception as err:
        print(f"Error exiting application:{err}")


def create_room_editor_widgets() -> tuple:
    """Create widgets for room editing functionality.

    Returns:
        Tuple containing all room editor widgets
    """
    try:
        room_list = widgets.Select(
            options=[(f"Room {room['RoomID']}: {room['Title']}", room["RoomID"]) for room in data_manager.rooms_data["rooms"]],
            description="Rooms:",
            layout=widgets.Layout(width="300px"),
        )

        room_id = widgets.IntText(description="Room ID:")
        room_title = widgets.Text(description="Title:")
        room_description = widgets.Textarea(description="Description:", layout=widgets.Layout(width="400px", height="100px"))

        new_room_button = widgets.Button(description="New Room")
        save_room_button = widgets.Button(description="Save Room")
        delete_room_button = widgets.Button(description="Delete Room")
        save_all_rooms_button = widgets.Button(description="Save All")

        output = widgets.Output()

        return (
            room_list,
            room_id,
            room_title,
            room_description,
            new_room_button,
            save_room_button,
            delete_room_button,
            save_all_rooms_button,
            output,
        )

    except Exception as err:
        print(f"Error creating room editor widgets:{err}")
        return tuple([widgets.HTML(f"Error:{err}") for _ in range(9)])


def on_room_select(change: dict, room_id: widgets.IntText, room_title: widgets.Text, room_description: widgets.Textarea) -> None:
    """Handle room selection from the list.

    Args:
        change: Widget change event data
        room_id: Room ID integer widget
        room_title: Room title text widget
        room_description: Room description textarea widget
    """
    try:
        if change["type"] == "change" and change["name"] == "value":
            selected_room = data_manager.find_room_by_id(change["new"])
            if selected_room and data_manager.validate_room_data(selected_room):
                room_id.value = selected_room["RoomID"]
                room_title.value = selected_room["Title"]
                room_description.value = selected_room.get("Description", "")
            else:
                print(f"Warning: Selected room {change['new']} not found or invalid")
    except Exception as err:
        print(f"Error selecting room:{err}")


def update_room_list(room_list: widgets.Select) -> None:
    """Update the room list widget with current room data.

    Args:
        room_list: Room selection widget to update
    """
    try:
        formatted_rooms = [
            (f"Room {room['RoomID']}: {room['Title']}", room["RoomID"])
            for room in data_manager.rooms_data["rooms"]
            if data_manager.validate_room_data(room)
        ]
        formatted_rooms.sort(key=lambda x: x[1])
        room_list.options = formatted_rooms

        if formatted_rooms and not room_list.value:
            room_list.value = formatted_rooms[0][1]
    except Exception as err:
        print(f"Error updating room list:{err}")


def on_new_room(
    b: widgets.Button,
    room_list: widgets.Select,
    room_id: widgets.IntText,
    room_title: widgets.Text,
    room_description: widgets.Textarea,
) -> None:
    """Create a new room.

    Args:
        b: Button widget that triggered the event
        room_list: Room selection widget
        room_id: Room ID integer widget
        room_title: Room title text widget
        room_description: Room description textarea widget
    """
    try:
        new_id = data_manager.get_next_room_id()

        new_room = {"RoomID": new_id, "Title": "New Room", "Description": "", "ExitID": []}

        if not data_manager.validate_room_data(new_room):
            print("Error: Could not create valid new room")
            return

        data_manager.rooms_data["rooms"].append(new_room)

        update_room_list(room_list)
        room_list.value = new_id
        on_room_select({"type": "change", "name": "value", "new": new_id}, room_id, room_title, room_description)
    except Exception as err:
        print(f"Error creating new room:{err}")


def on_save_room(
    b: widgets.Button,
    room_list: widgets.Select,
    room_id: widgets.IntText,
    room_title: widgets.Text,
    room_description: widgets.Textarea,
    output: widgets.Output,
) -> None:
    """Save changes to the selected room.

    Args:
        b: Button widget that triggered the event
        room_list: Room selection widget
        room_id: Room ID integer widget
        room_title: Room title text widget
        room_description: Room description textarea widget
        output: Output widget for displaying messages
    """
    try:
        selected_room = data_manager.find_room_by_id(room_id.value)
        if not selected_room:
            with output:
                clear_output()
                print(f"Error: Room {room_id.value} not found.")
            return

        # Validate title is not empty
        if not room_title.value.strip():
            with output:
                clear_output()
                print("Error: Room title cannot be empty.")
            return

        selected_room["Title"] = room_title.value.strip()
        selected_room["Description"] = room_description.value

        if not data_manager.validate_room_data(selected_room):
            with output:
                clear_output()
                print("Error: Room data validation failed after update.")
            return

        update_room_list(room_list)
        with output:
            clear_output()
            print(f"Room {room_id.value} updated successfully.")
    except Exception as err:
        with output:
            clear_output()
            print(f"Error saving room:{err}")


def on_delete_room(
    b: widgets.Button,
    room_list: widgets.Select,
    room_id: widgets.IntText,
    room_title: widgets.Text,
    room_description: widgets.Textarea,
    output: widgets.Output,
) -> None:
    """Delete the selected room and associated exits.

    Args:
        b: Button widget that triggered the event
        room_list: Room selection widget
        room_id: Room ID integer widget
        room_title: Room title text widget
        room_description: Room description textarea widget
        output: Output widget for displaying messages
    """
    try:
        room_to_delete = data_manager.find_room_by_id(room_id.value)
        if not room_to_delete:
            with output:
                clear_output()
                print(f"Error: Room {room_id.value} not found.")
            return

        # Remove all exits associated with this room
        exit_ids_to_remove = room_to_delete.get("ExitID", []).copy()
        for exit_id in exit_ids_to_remove:
            data_manager.exits_data["exits"] = [exit for exit in data_manager.exits_data["exits"] if exit["ExitID"] != exit_id]

        # Remove the room
        data_manager.rooms_data["rooms"] = [room for room in data_manager.rooms_data["rooms"] if room["RoomID"] != room_id.value]

        update_room_list(room_list)
        if data_manager.rooms_data["rooms"]:
            room_list.value = data_manager.rooms_data["rooms"][0]["RoomID"]
            on_room_select({"type": "change", "name": "value", "new": room_list.value}, room_id, room_title, room_description)
        else:
            clear_room_fields(room_id, room_title, room_description)

        with output:
            clear_output()
            print(f"Room {room_id.value} and its exits deleted successfully.")
    except Exception as err:
        with output:
            clear_output()
            print(f"Error deleting room:{err}")


def clear_room_fields(room_id: widgets.IntText, room_title: widgets.Text, room_description: widgets.Textarea) -> None:
    """Clear all room editing fields.

    Args:
        room_id: Room ID integer widget
        room_title: Room title text widget
        room_description: Room description textarea widget
    """
    try:
        room_id.value = 0
        room_title.value = ""
        room_description.value = ""
    except Exception as err:
        print(f"Error clearing room fields:{err}")

In [ ]:
def clear_exit_fields(
    exit_id: widgets.Text, direction: widgets.Text, target_room: widgets.IntText, visible: widgets.Checkbox
) -> None:
    """Clear all exit editing fields.

    Args:
        exit_id: Exit ID text widget
        direction: Direction text widget
        target_room: Target room integer widget
        visible: Visibility checkbox widget
    """
    try:
        exit_id.value = ""
        direction.value = ""
        target_room.value = 0
        visible.value = True
    except Exception as err:
        print(f"Error clearing exit fields:{err}")


def create_room_editor_tab() -> widgets.VBox:
    """Create the room editor tab.

    Returns:
        VBox widget containing the room editor interface
    """
    try:
        (
            room_list,
            room_id,
            room_title,
            room_description,
            new_room_button,
            save_room_button,
            delete_room_button,
            save_all_rooms_button,
            output,
        ) = create_room_editor_widgets()

        room_list.observe(lambda change: on_room_select(change, room_id, room_title, room_description), names="value")
        new_room_button.on_click(lambda b: on_new_room(b, room_list, room_id, room_title, room_description))
        save_room_button.on_click(lambda b: on_save_room(b, room_list, room_id, room_title, room_description, output))
        delete_room_button.on_click(lambda b: on_delete_room(b, room_list, room_id, room_title, room_description, output))
        save_all_rooms_button.on_click(lambda b: on_save_all(b, output))

        exit_app_button = widgets.Button(description="Exit Application", button_style="danger")
        exit_app_button.on_click(exit_app)

        # Initialize the room list
        update_room_list(room_list)
        if data_manager.rooms_data["rooms"]:
            on_room_select({"type": "change", "name": "value", "new": room_list.value}, room_id, room_title, room_description)

        return widgets.VBox(
            [
                widgets.HBox([room_list, widgets.VBox([room_id, room_title, room_description])]),
                widgets.HBox([new_room_button, save_room_button, delete_room_button, save_all_rooms_button, exit_app_button]),
                output,
            ]
        )

    except Exception as err:
        error_widget = widgets.HTML(f"<p>Error creating room editor tab:{err}</p>")
        return widgets.VBox([error_widget])

In [ ]:
def on_delete_exit(
    b: widgets.Button,
    exit_list: widgets.Select,
    exit_id: widgets.Text,
    direction: widgets.Text,
    target_room: widgets.IntText,
    visible: widgets.Checkbox,
    output: widgets.Output,
) -> None:
    """Delete the selected exit.

    Args:
        b: Button widget that triggered the event
        exit_list: Exit selection widget
        exit_id: Exit ID text widget
        direction: Direction text widget
        target_room: Target room integer widget
        visible: Visibility checkbox widget
        output: Output widget for displaying messages
    """
    try:
        exit_to_delete = data_manager.find_exit_by_id(exit_id.value)
        if not exit_to_delete:
            with output:
                clear_output()
                print(f"Exit {exit_id.value} not found.")
            return

        # Remove the exit from the data
        data_manager.exits_data["exits"] = [exit for exit in data_manager.exits_data["exits"] if exit["ExitID"] != exit_id.value]

        # Remove the exit from rooms_data
        data_manager.remove_exit_from_rooms(exit_id.value)

        update_exit_list(exit_list)

        # Select the next available exit or clear fields
        if data_manager.exits_data["exits"]:
            # Find first valid exit
            valid_exits = [
                exit
                for exit in data_manager.exits_data["exits"]
                if any(exit["ExitID"] in room.get("ExitID", []) for room in data_manager.rooms_data["rooms"])
            ]
            if valid_exits and exit_list.options:
                exit_list.value = valid_exits[0]["ExitID"]
                on_exit_select(
                    {"type": "change", "name": "value", "new": exit_list.value}, exit_id, direction, target_room, visible
                )
            else:
                clear_exit_fields(exit_id, direction, target_room, visible)
        else:
            clear_exit_fields(exit_id, direction, target_room, visible)

        with output:
            clear_output()
            print(f"Exit {exit_id.value} deleted successfully.")

    except Exception as err:
        with output:
            clear_output()
            print(f"Error deleting exit:{err}")

In [ ]:
# Placeholder cell - functions moved to cell 10

In [ ]:
# Placeholder cell - functions moved to cell 10

In [ ]:
def create_room_visualization_tab() -> widgets.VBox:
    """Create the room visualization tab.

    Returns:
        VBox widget containing the visualization interface
    """
    try:
        output = widgets.Output()
        image = widgets.Image()
        update_button = widgets.Button(description="Update Graph")
        exit_button = widgets.Button(description="Exit Application", button_style="danger")

        def update_graph(b=None) -> None:
            """Update the graph visualization.

            Args:
                b: Button widget that triggered the event (optional)
            """
            try:
                with output:
                    clear_output()
                    print("Updating graph visualization...")

                G = create_graph()  # Recreate the graph
                img_str = visualize_graph(G)
                image.value = base64.b64decode(img_str)

                with output:
                    clear_output()
                    print(f"Graph updated successfully. Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

            except RoomEditorError:
                with output:
                    clear_output()
                    print("Error updating graph")
            except Exception as err:
                with output:
                    clear_output()
                    print(f"Unexpected error updating graph:{err}")

        update_button.on_click(update_graph)
        exit_button.on_click(exit_app)

        # Store the update function for external access
        update_button.update_graph_func = update_graph

        room_viz_tab = widgets.VBox([widgets.HBox([update_button, exit_button]), image, output])

        update_graph()  # Initial graph update
        return room_viz_tab

    except Exception as err:
        error_widget = widgets.HTML(f"<p>Error creating visualization tab:{err}</p>")
        return widgets.VBox([error_widget])

In [ ]:
def create_exit_editor_tab() -> widgets.VBox:
    """Create the exit editor tab.

    Returns:
        VBox widget containing the exit editor interface
    """
    try:
        (exit_list, exit_id, direction, target_room, visible, new_button, save_button, delete_button, save_all_button, output) = (
            create_exit_editor_widgets()
        )

        exit_list.observe(lambda change: on_exit_select(change, exit_id, direction, target_room, visible), names="value")
        new_button.on_click(lambda b: on_new_exit(b, exit_list, exit_id, direction, target_room, visible))
        save_button.on_click(lambda b: on_save_exit(b, exit_list, exit_id, direction, target_room, visible, output))
        delete_button.on_click(lambda b: on_delete_exit(b, exit_list, exit_id, direction, target_room, visible, output))
        save_all_button.on_click(lambda b: on_save_all(b, output))

        exit_app_button = widgets.Button(description="Exit Application", button_style="danger")
        exit_app_button.on_click(exit_app)

        # Initialize the exit list
        update_exit_list(exit_list)
        if data_manager.exits_data["exits"]:
            valid_exits = [
                exit
                for exit in data_manager.exits_data["exits"]
                if any(exit["ExitID"] in room.get("ExitID", []) for room in data_manager.rooms_data["rooms"])
            ]
            if valid_exits and exit_list.options:
                on_exit_select(
                    {"type": "change", "name": "value", "new": exit_list.value}, exit_id, direction, target_room, visible
                )

        return widgets.VBox(
            [
                widgets.HBox([exit_list, widgets.VBox([exit_id, direction, target_room, visible])]),
                widgets.HBox([new_button, save_button, delete_button, save_all_button, exit_app_button]),
                output,
            ]
        )

    except Exception as err:
        error_widget = widgets.HTML(f"<p>Error creating exit editor tab:{err}</p>")
        return widgets.VBox([error_widget])

In [ ]:
def room_editor_app() -> widgets.VBox:
    """Main application layout with room visualization, exit editor, and room editor tabs.

    Returns:
        VBox widget containing the complete application interface

    Raises:
        RoomEditorError: If application initialization fails
    """
    try:
        room_viz_tab = create_room_visualization_tab()
        exit_editor_tab = create_exit_editor_tab()
        room_editor_tab = create_room_editor_tab()

        tab = Tab(children=[room_viz_tab, exit_editor_tab, room_editor_tab])
        tab.set_title(0, "Room Visualization")
        tab.set_title(1, "Exit Editor")
        tab.set_title(2, "Room Editor")

        def on_tab_change(change: dict) -> None:
            """Handle tab changes to refresh visualizations.

            Args:
                change: Widget change event data
            """
            try:
                if change["new"] == 0:  # Room Visualization tab
                    # Access the update function directly instead of simulating button click
                    update_button = room_viz_tab.children[0].children[0]
                    if hasattr(update_button, "update_graph_func"):
                        update_button.update_graph_func()
            except Exception as err:
                print(f"Error handling tab change:{err}")

        tab.observe(on_tab_change, names="selected_index")

        app_layout = widgets.VBox([widgets.HTML("<h1>Room Editor</h1>"), tab])

        return app_layout

    except Exception as err:
        error_widget = widgets.HTML(f"<h1>Error Loading Room Editor</h1><p>{err}</p>")
        return widgets.VBox([error_widget])

In [ ]:
# Display the app with error handling
try:
    display(room_editor_app())
except Exception as err:
    print(f"Critical error starting Room Editor:{err}")
    display(widgets.HTML(f"<h1>Critical Error</h1><p>Failed to start Room Editor:{err}</p>"))